In [11]:
ravedess_path='ravedess-emotional-speech-audio'
!pip list

Package                   Version
------------------------- -----------
anyio                     4.12.1
argon2-cffi               25.1.0
argon2-cffi-bindings      25.1.0
arrow                     1.4.0
asttokens                 3.0.1
async-lru                 2.1.0
attrs                     25.4.0
audioop-lts               0.2.2
audioread                 3.1.0
babel                     2.18.0
beautifulsoup4            4.14.3
bleach                    6.3.0
certifi                   2026.1.4
cffi                      2.0.0
charset-normalizer        3.4.4
colorama                  0.4.6
comm                      0.2.3
contourpy                 1.3.3
cycler                    0.12.1
debugpy                   1.8.20
decorator                 5.2.1
defusedxml                0.7.1
executing                 2.2.1
fastjsonschema            2.21.2
filelock                  3.24.0
fonttools                 4.61.1
fqdn                      1.5.1
fsspec                    2026.2.0
h11            

In [12]:
# import os
# import pandas as pd
# import librosa
# import numpy as np

# # -------------------------------------------------
# # Create RAVDESS dataset (paths + emotion labels)
# # -------------------------------------------------
# def create_ravdess_dataset(directory_path):

#     emotion_map = {
#         '01': 'neutral',
#         '02': 'calm',
#         '03': 'happy',
#         '04': 'sad',
#         '05': 'angry',
#         '06': 'fearful',
#         '07': 'disgust',
#         '08': 'surprised'
#     }

#     dataset = []

#     for subdir, _, files in os.walk(directory_path):
#         for file in files:
#             if file.lower().endswith(".wav"):

#                 parts = file.split("-")
#                 if len(parts) < 7:
#                     continue

#                 vocal_channel = parts[1]   # 01 = speech, 02 = song
#                 emotion_code  = parts[2]   # emotion code

#                 # keep only speech samples
#                 if vocal_channel != "01":
#                     continue

#                 if emotion_code in emotion_map:
#                     dataset.append({
#                         "path": os.path.join(subdir, file),
#                         "emotion": emotion_map[emotion_code]
#                     })

#     return dataset


# # -------------------------------------------------
# # Feature extraction (MFCC + Chroma)
# # -------------------------------------------------
# def extract_features(df, n_mfcc=13):

#     features = []

#     for idx, row in df.iterrows():
#         try:
#             if not os.path.exists(row["path"]):
#                 print(f"❌ File not found: {row['path']}")
#                 continue

#             y, sr = librosa.load(row["path"], sr=16000)

#             if len(y) == 0:
#                 print(f"❌ Empty audio: {row['path']}")
#                 continue

#             # MFCCs
#             mfccs = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=n_mfcc)
#             mfccs_mean = np.mean(mfccs, axis=1)

#             # Chroma
#             chroma = librosa.feature.chroma_stft(y=y, sr=sr)
#             chroma_mean = np.mean(chroma, axis=1)

#             features.append({
#                 "mfccs": mfccs_mean,
#                 "chroma": chroma_mean,
#                 "emotion": row["emotion"]
#             })

#             print(f"✅ Processed {idx+1}: {os.path.basename(row['path'])}")

#         except Exception as e:
#             print(f"❌ Error processing {row['path']}: {e}")

#     return features


# # -------------------------------------------------
# # MAIN SCRIPT
# # -------------------------------------------------

# # ⚠️ UPDATE THIS PATH IF NEEDED
# ravdess_path = "ravedess-emotional-speech-audio"

# # Create dataset
# ravdess_data = create_ravdess_dataset(ravdess_path)
# df_ravdess = pd.DataFrame(ravdess_data)

# print("📊 Total samples found:", len(df_ravdess))

# if df_ravdess.empty:
#     raise RuntimeError("❌ Dataset is empty — check path or dataset structure")

# print(df_ravdess["emotion"].value_counts())

# # Feature extraction
# extracted_features = extract_features(df_ravdess)

# # -------------------------------------------------
# # Flatten features into ML-ready DataFrame
# # -------------------------------------------------
# flattened_features = []

# for item in extracted_features:
#     row = {"emotion": item["emotion"]}

#     for i, v in enumerate(item["mfccs"]):
#         row[f"mfcc_{i}"] = v

#     for i, v in enumerate(item["chroma"]):
#         row[f"chroma_{i}"] = v

#     flattened_features.append(row)

# df_features = pd.DataFrame(flattened_features)

# # Save extracted features
# csv_filename = "extracted_audio_features.csv"
# df_features.to_csv(csv_filename, index=False)

# print(f"\n✅ Feature extraction complete")
# print(f"📁 Saved to: {csv_filename}")
# print("📐 Shape:", df_features.shape)


In [13]:
import os
import numpy as np
import librosa

# --------------------------------------------------
# Emotion mapping (RAVDESS standard)
# --------------------------------------------------
EMOTION_MAP = {
    "01": "neutral",
    "02": "calm",
    "03": "happy",
    "04": "sad",
    "05": "angry",
    "06": "fearful",
    "07": "disgust",
    "08": "surprised"
}

# --------------------------------------------------
# Helper: frequency-axis padding
# --------------------------------------------------
def pad_freq(x, target_F):
    """
    Pads frequency axis (axis=0) to target_F
    x: (F, T)
    """
    F, T = x.shape
    if F == target_F:
        return x
    pad_amt = target_F - F
    return np.pad(x, ((0, pad_amt), (0, 0)), mode="constant")


# --------------------------------------------------
# Multi-Channel Spectral Feature Extraction
# --------------------------------------------------
def extract_multichannel_spectral_features(
    wav_path,
    sr=16000,
    n_mfcc=40,
    n_mels=64
):
    """
    Input  : path to a single .wav file
    Output : Tensor (C=3, F=64, T)
    """

    # 1️⃣ Load raw audio
    y, sr = librosa.load(wav_path, sr=sr, mono=True)
    if len(y) == 0:
        raise ValueError("Empty audio signal")

    # 2️⃣ Spectral features
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=n_mfcc)          # (40, T)
    mel = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=n_mels) # (64, T)
    chroma = librosa.feature.chroma_stft(y=y, sr=sr)               # (12, T)

    # 3️⃣ Normalize per channel
    def norm(x):
        return (x - x.mean()) / (x.std() + 1e-6)

    mfcc = norm(mfcc)
    mel = norm(mel)
    chroma = norm(chroma)

    # 4️⃣ Time alignment
    T = min(mfcc.shape[1], mel.shape[1], chroma.shape[1])
    mfcc = mfcc[:, :T]
    mel = mel[:, :T]
    chroma = chroma[:, :T]

    # 5️⃣ Frequency alignment (CRITICAL FIX)
    target_F = max(mfcc.shape[0], mel.shape[0], chroma.shape[0])
    mfcc = pad_freq(mfcc, target_F)
    mel = pad_freq(mel, target_F)
    chroma = pad_freq(chroma, target_F)

    # 6️⃣ Stack channels
    X = np.stack([mfcc, mel, chroma], axis=0)
    # Shape: (3, 64, T)

    return X


# --------------------------------------------------
# Load RAVDESS dataset from ROOT directory
# --------------------------------------------------
def load_ravdess_dataset(dataset_root):
    """
    Input  : ravdess-emotional-speech-audio/
    Output : list of (feature_tensor, emotion_label)
    """

    data = []

    for subdir, _, files in os.walk(dataset_root):
        for file in files:
            if not file.endswith(".wav"):
                continue

            parts = file.split("-")
            if len(parts) < 7:
                continue

            vocal_channel = parts[1]   # 01 = speech
            emotion_code = parts[2]

            if vocal_channel != "01":
                continue
            if emotion_code not in EMOTION_MAP:
                continue

            wav_path = os.path.join(subdir, file)
            emotion = EMOTION_MAP[emotion_code]

            try:
                X = extract_multichannel_spectral_features(wav_path)
                data.append((X, emotion))
                print(f"✅ Loaded: {file} → {emotion} | shape={X.shape}")

            except Exception as e:
                print(f"❌ Error processing {file}: {e}")

    return data


# --------------------------------------------------
# MAIN
# --------------------------------------------------
if __name__ == "__main__":

    # 🔴 DATASET ROOT (as you specified)
    DATASET_ROOT = "ravedess-emotional-speech-audio"

    dataset = load_ravdess_dataset(DATASET_ROOT)

    print("\n📊 Total samples loaded:", len(dataset))

    # Sanity check
    X, y = dataset[0]
    print("🔹 One sample tensor shape:", X.shape)
    print("🔹 Emotion label:", y)


✅ Loaded: 03-01-01-01-01-01-01.wav → neutral | shape=(3, 64, 104)
✅ Loaded: 03-01-01-01-01-02-01.wav → neutral | shape=(3, 64, 105)
✅ Loaded: 03-01-01-01-02-01-01.wav → neutral | shape=(3, 64, 103)
✅ Loaded: 03-01-01-01-02-02-01.wav → neutral | shape=(3, 64, 100)
✅ Loaded: 03-01-02-01-01-01-01.wav → calm | shape=(3, 64, 111)
✅ Loaded: 03-01-02-01-01-02-01.wav → calm | shape=(3, 64, 113)
✅ Loaded: 03-01-02-01-02-01-01.wav → calm | shape=(3, 64, 110)
✅ Loaded: 03-01-02-01-02-02-01.wav → calm | shape=(3, 64, 109)
✅ Loaded: 03-01-02-02-01-01-01.wav → calm | shape=(3, 64, 116)
✅ Loaded: 03-01-02-02-01-02-01.wav → calm | shape=(3, 64, 126)
✅ Loaded: 03-01-02-02-02-01-01.wav → calm | shape=(3, 64, 132)
✅ Loaded: 03-01-02-02-02-02-01.wav → calm | shape=(3, 64, 126)
✅ Loaded: 03-01-03-01-01-01-01.wav → happy | shape=(3, 64, 109)
✅ Loaded: 03-01-03-01-01-02-01.wav → happy | shape=(3, 64, 109)
✅ Loaded: 03-01-03-01-02-01-01.wav → happy | shape=(3, 64, 111)
✅ Loaded: 03-01-03-01-02-02-01.wav → hap

In [14]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class MultiScaleCNN(nn.Module):
    def __init__(
        self,
        in_channels=3,
        base_channels=32,
        out_channels=128
    ):
        super().__init__()

        # ----- Multi-scale branches -----
        self.branch_3x3 = nn.Conv2d(
            in_channels, base_channels,
            kernel_size=3, padding=1
        )

        self.branch_5x5 = nn.Conv2d(
            in_channels, base_channels,
            kernel_size=5, padding=2
        )

        self.branch_7x7 = nn.Conv2d(
            in_channels, base_channels,
            kernel_size=7, padding=3
        )

        # ----- Feature fusion -----
        self.bn = nn.BatchNorm2d(base_channels * 3)

        self.proj = nn.Conv2d(
            base_channels * 3,
            out_channels,
            kernel_size=1
        )

    def forward(self, x):
        """
        x: (B, 3, F=64, T)
        """

        x1 = self.branch_3x3(x)
        x2 = self.branch_5x5(x)
        x3 = self.branch_7x7(x)

        # Concatenate along channel dimension
        x = torch.cat([x1, x2, x3], dim=1)

        x = self.bn(x)
        x = F.relu(x)

        # Channel projection
        x = self.proj(x)

        return x


In [15]:
# --------------------------------------------------
# CNN → Sequence projection
# (Required before Conformer / Attention)
# --------------------------------------------------
def cnn_to_sequence(x):
    """
    Input :
        x : (B, D, F, T)  ← output of MultiScaleCNN
    Output:
        seq : (B, T, D)   ← sequence for attention models
    """
    # Collapse frequency axis
    x = x.mean(dim=2)      # (B, D, T)

    # Rearrange to (batch, time, features)
    x = x.permute(0, 2, 1) # (B, T, D)

    return x


# --------------------------------------------------
# (Optional) Simple sanity test
# --------------------------------------------------
if __name__ == "__main__":

    # Fake batch for testing
    B, T = 2, 300
    X = torch.randn(B, 3, 64, T)

    # Multi-scale CNN
    mscnn = MultiScaleCNN(
        in_channels=3,
        base_channels=32,
        out_channels=128
    )

    cnn_out = mscnn(X)

    # Convert to sequence
    seq = cnn_to_sequence(cnn_out)

    print("Input shape      :", X.shape)        # (2, 3, 64, 300)
    print("CNN output shape :", cnn_out.shape)  # (2, 128, 64, 300)
    print("Sequence shape   :", seq.shape)      # (2, 300, 128)


Input shape      : torch.Size([2, 3, 64, 300])
CNN output shape : torch.Size([2, 128, 64, 300])
Sequence shape   : torch.Size([2, 300, 128])


In [16]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# ==================================================
# Feed Forward Module (Conformer FFN)
# ==================================================
class FeedForward(nn.Module):
    def __init__(self, dim, expansion=4, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.LayerNorm(dim),
            nn.Linear(dim, dim * expansion),
            nn.SiLU(),
            nn.Dropout(dropout),
            nn.Linear(dim * expansion, dim),
            nn.Dropout(dropout)
        )

    def forward(self, x):
        # x: (B, T, D)
        return self.net(x)


# ==================================================
# Conformer Convolution Module (Local modeling)
# ==================================================
class ConformerConvModule(nn.Module):
    def __init__(self, dim, kernel_size=31, dropout=0.1):
        super().__init__()

        self.layer_norm = nn.LayerNorm(dim)

        self.pointwise_conv1 = nn.Conv1d(dim, dim * 2, kernel_size=1)
        self.glu = nn.GLU(dim=1)

        self.depthwise_conv = nn.Conv1d(
            dim,
            dim,
            kernel_size=kernel_size,
            padding=kernel_size // 2,
            groups=dim
        )

        self.batch_norm = nn.BatchNorm1d(dim)
        self.activation = nn.SiLU()

        self.pointwise_conv2 = nn.Conv1d(dim, dim, kernel_size=1)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        # x: (B, T, D)
        x = self.layer_norm(x)
        x = x.transpose(1, 2)      # (B, D, T)

        x = self.pointwise_conv1(x)
        x = self.glu(x)

        x = self.depthwise_conv(x)
        x = self.batch_norm(x)
        x = self.activation(x)

        x = self.pointwise_conv2(x)
        x = self.dropout(x)

        x = x.transpose(1, 2)      # (B, T, D)
        return x


# ==================================================
# Single Conformer Block
# ==================================================
class ConformerBlock(nn.Module):
    def __init__(
        self,
        dim,
        num_heads=4,
        ff_expansion=4,
        conv_kernel=31,
        dropout=0.1
    ):
        super().__init__()

        self.ff1 = FeedForward(dim, ff_expansion, dropout)

        self.self_attn = nn.MultiheadAttention(
            embed_dim=dim,
            num_heads=num_heads,
            dropout=dropout,
            batch_first=True
        )

        self.conv = ConformerConvModule(dim, conv_kernel, dropout)
        self.ff2 = FeedForward(dim, ff_expansion, dropout)

        self.norm = nn.LayerNorm(dim)

    def forward(self, x, attn_mask=None):
        # x: (B, T, D)

        # FFN (half step)
        x = x + 0.5 * self.ff1(x)

        # Multi-Head Self Attention
        attn_out, _ = self.self_attn(x, x, x, attn_mask=attn_mask)
        x = x + attn_out

        # Convolution Module
        x = x + self.conv(x)

        # FFN (half step)
        x = x + 0.5 * self.ff2(x)

        # Final normalization
        x = self.norm(x)

        return x


# ==================================================
# Conformer Encoder (Stacked Blocks)
# ==================================================
class ConformerEncoder(nn.Module):
    def __init__(
        self,
        dim=128,
        num_layers=4,
        num_heads=4,
        dropout=0.1
    ):
        super().__init__()

        self.layers = nn.ModuleList([
            ConformerBlock(
                dim=dim,
                num_heads=num_heads,
                dropout=dropout
            )
            for _ in range(num_layers)
        ])

    def forward(self, x, attn_mask=None):
        # x: (B, T, D)
        for layer in self.layers:
            x = layer(x, attn_mask)
        return x


# ==================================================
# SANITY TEST (RUN THIS)
# ==================================================
if __name__ == "__main__":
    B, T, D = 2, 300, 128
    x = torch.randn(B, T, D)

    encoder = ConformerEncoder(dim=D, num_layers=4)
    y = encoder(x)

    print("Input shape :", x.shape)
    print("Output shape:", y.shape)


Input shape : torch.Size([2, 300, 128])
Output shape: torch.Size([2, 300, 128])


In [17]:
#SQUASH
import torch
import torch.nn as nn
import torch.nn.functional as F

def squash(v, dim=-1, eps=1e-8):
    """
    Squashing non-linearity for capsules
    """
    norm_sq = (v ** 2).sum(dim=dim, keepdim=True)
    norm = torch.sqrt(norm_sq + eps)
    return (norm_sq / (1.0 + norm_sq)) * (v / (norm + eps))


In [18]:
#PrimaryTemporalCapsules
class PrimaryTemporalCapsules(nn.Module):
    def __init__(
        self,
        input_dim,          # D from Conformer (e.g., 128)
        num_capsules=16,
        capsule_dim=16
    ):
        super().__init__()
        self.num_capsules = num_capsules
        self.capsule_dim = capsule_dim

        self.proj = nn.Linear(input_dim, num_capsules * capsule_dim)

    def forward(self, x):
        """
        x: (B, T, D)
        return: (B, N_primary, capsule_dim)
        """
        B, T, _ = x.shape

        u = self.proj(x)                                # (B, T, Np*Cd)
        u = u.view(B, T, self.num_capsules, self.capsule_dim)

        # Temporal aggregation (emotion unfolds over time)
        u = u.mean(dim=1)                               # (B, Np, Cd)
        u = squash(u)

        return u


In [19]:
#DynamicRoutingCapsules
class DynamicRouting(nn.Module):
    def __init__(
        self,
        num_input_caps,
        num_output_caps,
        input_dim,
        output_dim,
        routing_iters=3
    ):
        super().__init__()
        self.routing_iters = routing_iters

        self.W = nn.Parameter(
            0.01 * torch.randn(
                1, num_input_caps, num_output_caps, output_dim, input_dim
            )
        )

    def forward(self, u):
        """
        u: (B, N_in, input_dim)
        return: (B, N_out, output_dim)
        """
        B, N_in, _ = u.shape
        N_out = self.W.size(2)

        u = u.unsqueeze(2).unsqueeze(-1)                 # (B, N_in, 1, input_dim, 1)
        W = self.W.expand(B, -1, -1, -1, -1)

        u_hat = torch.matmul(W, u).squeeze(-1)           # (B, N_in, N_out, output_dim)

        b = torch.zeros(B, N_in, N_out, device=u.device)

        for _ in range(self.routing_iters):
            c = F.softmax(b, dim=-1)
            s = (c.unsqueeze(-1) * u_hat).sum(dim=1)
            v = squash(s)

            b = b + (u_hat * v.unsqueeze(1)).sum(dim=-1)

        return v


In [20]:
class EmotionCapsules(nn.Module):
    def __init__(
        self,
        num_primary_caps,
        primary_dim,
        num_emotions=8,
        emotion_dim=16,
        routing_iters=3
    ):
        super().__init__()

        self.routing = DynamicRouting(
            num_input_caps=num_primary_caps,
            num_output_caps=num_emotions,
            input_dim=primary_dim,
            output_dim=emotion_dim,
            routing_iters=routing_iters
        )

    def forward(self, u):
        """
        u: (B, N_primary, primary_dim)
        """
        v = self.routing(u)
        probs = torch.norm(v, dim=-1)    # capsule length = emotion probability
        return v, probs


In [21]:
#DropInModule
class SERCapsuleNetwork(nn.Module):
    def __init__(
        self,
        conformer_dim=128,
        num_primary_caps=16,
        primary_dim=16,
        num_emotions=8,
        emotion_dim=16
    ):
        super().__init__()

        self.primary_caps = PrimaryTemporalCapsules(
            input_dim=conformer_dim,
            num_capsules=num_primary_caps,
            capsule_dim=primary_dim
        )

        self.emotion_caps = EmotionCapsules(
            num_primary_caps=num_primary_caps,
            primary_dim=primary_dim,
            num_emotions=num_emotions,
            emotion_dim=emotion_dim
        )

    def forward(self, x):
        """
        x: (B, T, D) from Conformer
        """
        u = self.primary_caps(x)
        v, probs = self.emotion_caps(u)
        return v, probs


In [22]:
#MarginLoss
class CapsuleMarginLoss(nn.Module):
    def __init__(self, m_plus=0.9, m_minus=0.1, lambda_=0.5):
        super().__init__()
        self.m_plus = m_plus
        self.m_minus = m_minus
        self.lambda_ = lambda_

    def forward(self, probs, labels):
        """
        probs : (B, N_emotions)
        labels: (B,)
        """
        one_hot = F.one_hot(labels, probs.size(1)).float()

        loss_pos = one_hot * F.relu(self.m_plus - probs).pow(2)
        loss_neg = (1 - one_hot) * F.relu(probs - self.m_minus).pow(2)

        loss = loss_pos + self.lambda_ * loss_neg
        return loss.sum(dim=1).mean()


In [23]:
if __name__ == "__main__":
    B, T, D = 2, 300, 128
    x = torch.randn(B, T, D)

    caps = SERCapsuleNetwork(conformer_dim=D)
    v, probs = caps(x)

    print("Emotion capsules:", v.shape)    # (2, 8, 16)
    print("Emotion probs   :", probs.shape)


Emotion capsules: torch.Size([2, 8, 16])
Emotion probs   : torch.Size([2, 8])


In [24]:
import torch
import torch.nn as nn

class MSA_ECN(nn.Module):
    def __init__(
        self,
        num_emotions=8,
        cnn_out_dim=128,
        conformer_layers=4
    ):
        super().__init__()

        self.cnn = MultiScaleCNN(
            in_channels=3,
            base_channels=32,
            out_channels=cnn_out_dim
        )

        self.conformer = ConformerEncoder(
            dim=cnn_out_dim,
            num_layers=conformer_layers
        )

        self.capsules = SERCapsuleNetwork(
            conformer_dim=cnn_out_dim,
            num_emotions=num_emotions
        )

    def forward(self, x):
        """
        x: (B, 3, 64, T)
        """
        x = self.cnn(x)                 # (B, D, F, T)
        x = cnn_to_sequence(x)          # (B, T, D)
        x = self.conformer(x)           # (B, T, D)
        v, probs = self.capsules(x)     # capsules + probabilities
        return v, probs

In [25]:
from torch.utils.data import Dataset

class RAVDESSDataset(Dataset):
    def __init__(self, data, emotion_to_idx):
        self.data = data
        self.emotion_to_idx = emotion_to_idx

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        X, emotion = self.data[idx]
        X = torch.tensor(X, dtype=torch.float32)
        y = torch.tensor(self.emotion_to_idx[emotion], dtype=torch.long)
        return X, y

In [26]:
emotion_to_idx = {
    "neutral": 0,
    "calm": 1,
    "happy": 2,
    "sad": 3,
    "angry": 4,
    "fearful": 5,
    "disgust": 6,
    "surprised": 7
}

In [27]:
def train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0.0

    for X, y in loader:
        X, y = X.to(device), y.to(device)

        optimizer.zero_grad()

        _, probs = model(X)
        loss = criterion(probs, y)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)

In [28]:
def eval_epoch(model, loader, device):
    model.eval()
    correct, total = 0, 0

    with torch.no_grad():
        for X, y in loader:
            X, y = X.to(device), y.to(device)
            _, probs = model(X)

            preds = probs.argmax(dim=1)
            correct += (preds == y).sum().item()
            total += y.size(0)

    return correct / total

In [30]:
# =========================================================
# FULL UPDATED TRAINING BLOCK (VARIABLE-LENGTH SAFE)
# =========================================================

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

# ---------------------------------------------------------
# Dataset
# ---------------------------------------------------------
class RAVDESSDataset(Dataset):
    def __init__(self, data, emotion_to_idx):
        self.data = data
        self.emotion_to_idx = emotion_to_idx

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        X, emotion = self.data[idx]
        X = torch.tensor(X, dtype=torch.float32)
        y = torch.tensor(self.emotion_to_idx[emotion], dtype=torch.long)
        return X, y


# ---------------------------------------------------------
# Collate function (CRITICAL FIX)
# ---------------------------------------------------------
def pad_collate_fn(batch):
    """
    Pads variable-length spectrograms along time axis
    """
    Xs, ys = zip(*batch)

    max_T = max(x.shape[-1] for x in Xs)

    padded_X = []
    for x in Xs:
        pad_T = max_T - x.shape[-1]
        x = F.pad(x, (0, pad_T))      # pad time dimension
        padded_X.append(x)

    X = torch.stack(padded_X)         # (B, 3, 64, T_max)
    y = torch.tensor(ys)

    return X, y


# ---------------------------------------------------------
# Full Model Wrapper
# ---------------------------------------------------------
class MSA_ECN(nn.Module):
    def __init__(
        self,
        num_emotions=8,
        cnn_out_dim=128,
        conformer_layers=4
    ):
        super().__init__()

        self.cnn = MultiScaleCNN(
            in_channels=3,
            base_channels=32,
            out_channels=cnn_out_dim
        )

        self.conformer = ConformerEncoder(
            dim=cnn_out_dim,
            num_layers=conformer_layers
        )

        self.capsules = SERCapsuleNetwork(
            conformer_dim=cnn_out_dim,
            num_emotions=num_emotions
        )

    def forward(self, x):
        """
        x: (B, 3, 64, T)
        """
        x = self.cnn(x)                # (B, D, F, T)
        x = cnn_to_sequence(x)         # (B, T, D)
        x = self.conformer(x)          # (B, T, D)
        v, probs = self.capsules(x)    # capsule outputs
        return v, probs


# ---------------------------------------------------------
# Training / Evaluation
# ---------------------------------------------------------
def train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0.0

    for X, y in loader:
        X, y = X.to(device), y.to(device)

        optimizer.zero_grad()
        _, probs = model(X)
        loss = criterion(probs, y)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)


def eval_epoch(model, loader, device):
    model.eval()
    correct, total = 0, 0

    with torch.no_grad():
        for X, y in loader:
            X, y = X.to(device), y.to(device)
            _, probs = model(X)

            preds = probs.argmax(dim=1)
            correct += (preds == y).sum().item()
            total += y.size(0)

    return correct / total


# ---------------------------------------------------------
# MAIN
# ---------------------------------------------------------
if __name__ == "__main__":

    device = "cuda" if torch.cuda.is_available() else "cpu"

    emotion_to_idx = {
        "neutral": 0,
        "calm": 1,
        "happy": 2,
        "sad": 3,
        "angry": 4,
        "fearful": 5,
        "disgust": 6,
        "surprised": 7
    }

    # Load dataset (your existing function)
    dataset = load_ravdess_dataset(DATASET_ROOT)

    # Simple split (replace with speaker-independent later)
    split = int(0.8 * len(dataset))
    train_data = dataset[:split]
    test_data = dataset[split:]

    train_ds = RAVDESSDataset(train_data, emotion_to_idx)
    test_ds  = RAVDESSDataset(test_data, emotion_to_idx)

    train_loader = DataLoader(
        train_ds,
        batch_size=8,
        shuffle=True,
        collate_fn=pad_collate_fn
    )

    test_loader = DataLoader(
        test_ds,
        batch_size=8,
        collate_fn=pad_collate_fn
    )

    model = MSA_ECN(num_emotions=8).to(device)

    criterion = CapsuleMarginLoss()
    optimizer = optim.AdamW(
        model.parameters(),
        lr=3e-4,
        weight_decay=1e-4
    )

    # Training loop
    for epoch in range(1, 31):
        train_loss = train_epoch(
            model, train_loader, optimizer, criterion, device
        )
        acc = eval_epoch(model, test_loader, device)

        print(
            f"Epoch {epoch:02d} | "
            f"Loss: {train_loss:.4f} | "
            f"Acc: {acc:.4f}"
        )

    # Final sanity check
    X, y = next(iter(train_loader))
    X = X.to(device)

    with torch.no_grad():
        v, probs = model(X)

    print("Capsules:", v.shape)   # (B, 8, 16)
    print("Probs   :", probs.shape)

✅ Loaded: 03-01-01-01-01-01-01.wav → neutral | shape=(3, 64, 104)
✅ Loaded: 03-01-01-01-01-02-01.wav → neutral | shape=(3, 64, 105)
✅ Loaded: 03-01-01-01-02-01-01.wav → neutral | shape=(3, 64, 103)
✅ Loaded: 03-01-01-01-02-02-01.wav → neutral | shape=(3, 64, 100)
✅ Loaded: 03-01-02-01-01-01-01.wav → calm | shape=(3, 64, 111)
✅ Loaded: 03-01-02-01-01-02-01.wav → calm | shape=(3, 64, 113)
✅ Loaded: 03-01-02-01-02-01-01.wav → calm | shape=(3, 64, 110)
✅ Loaded: 03-01-02-01-02-02-01.wav → calm | shape=(3, 64, 109)
✅ Loaded: 03-01-02-02-01-01-01.wav → calm | shape=(3, 64, 116)
✅ Loaded: 03-01-02-02-01-02-01.wav → calm | shape=(3, 64, 126)
✅ Loaded: 03-01-02-02-02-01-01.wav → calm | shape=(3, 64, 132)
✅ Loaded: 03-01-02-02-02-02-01.wav → calm | shape=(3, 64, 126)
✅ Loaded: 03-01-03-01-01-01-01.wav → happy | shape=(3, 64, 109)
✅ Loaded: 03-01-03-01-01-02-01.wav → happy | shape=(3, 64, 109)
✅ Loaded: 03-01-03-01-02-01-01.wav → happy | shape=(3, 64, 111)
✅ Loaded: 03-01-03-01-02-02-01.wav → hap

In [31]:
torch.save(model.state_dict(),"msa_ecn_final.pth")
print("Model Saved as msa_ecn_final.pth")

Model Saved as msa_ecn_final.pth
